In [41]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 63.7 MB/s eta 0:00:00:00:0100:01


In [10]:

import fitz  # PyMuPDF
PYMUPDF_AVAILABLE = True


In [12]:
import pytesseract
from PIL import Image
import io as _io
OCR_AVAILABLE = True


 
OCR_LANGS = 'eng+san'  # English + Sanskrit (Devanagari). Requires tesseract-ocr-san installed.
OCR_MIN_CHARS = 20     # if native extraction yields fewer chars than this, treat page as scanned
OCR_DPI = 300          # higher = better accuracy on diacritics/conjuncts, but slower

In [13]:
def ocr_page(page, dpi=OCR_DPI, lang=OCR_LANGS):
    """Render a PDF page to an image and OCR it. Used only when native text extraction fails."""
    if not OCR_AVAILABLE:
        return ''
    try:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.open(_io.BytesIO(pix.tobytes("png")))
        return pytesseract.image_to_string(img, lang=lang)
    except Exception as e:
        print(f"OCR failed on a page: {e}")
        return ''

In [15]:
import os
import re
!pip install pdfplumber
import pdfplumber

from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 91.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.


In [16]:
!pip install tqdm
from tqdm import tqdm
TQDM_AVAILABLE = True

In [51]:
import pdfplumber

In [21]:
def extract_pdf_pages(file_path):
    """
    Extract text per page from a real PDF.
    Prefers PyMuPDF (fitz) - much faster than pdfplumber, since it skips
    the heavy character-level layout analysis and just pulls text.
    Falls back to pdfplumber if PyMuPDF isn't installed.
    Returns list of (page_num, page_text, is_estimated=False, ocr_used) - page numbers
    here are the ACTUAL PDF page numbers, not estimates.
    """
    if PYMUPDF_AVAILABLE:
        pages = []
        try:
            doc = fitz.open(file_path)
            for i, page in enumerate(doc, start=1):
                page_text = page.get_text() or ''
                ocr_used = False
 
                if len(page_text.strip()) < OCR_MIN_CHARS:
                    # Likely a scanned image page - fall back to OCR
                    ocr_text = ocr_page(page)
                    if len(ocr_text.strip()) > len(page_text.strip()):
                        page_text = ocr_text
                        ocr_used = True
 
                if page_text.strip():
                    pages.append((i, page_text, False, ocr_used))
                    print(pages)
                    
            doc.close()
        except Exception as e:
            print(f"Could not extract PDF {file_path} with PyMuPDF: {e}")
            return []
        return pages
         
    if PDFPLUMBER_AVAILABLE:
        pages = []
        try:
            with pdfplumber.open(file_path) as pdf:
                for i, page in enumerate(pdf.pages, start=1):
                    page_text = page.extract_text() or ''
                    if page_text.strip():
                        pages.append((i, page_text, False, False))
                        print(pages)
        except Exception as e:
            print(f"Could not extract PDF {file_path} with pdfplumber: {e}")
            return []
        return pages
 
    print("No PDF library installed - run `pip install pymupdf` (recommended) or `pip install pdfplumber`.")
    return []
 

In [22]:
extract_pdf_pages('/kaggle/input/datasets/rcratos/ayurveda-texts-english/Ayurveda Dataset/ayurveda_books/Charaka_Samhita_Text_with_English_Tanslation_-_P.V._Sharma.pdf')

[(1, 're ae\n4\n\nSAMHITA\n\nENGLISH TRANSLATION)\n\nP. V. SHARMA\n\nCHAUKHAMBHA ORIENTALIA\nVARANASI :\n\n \n\x0c', False, True)]
[(1, 're ae\n4\n\nSAMHITA\n\nENGLISH TRANSLATION)\n\nP. V. SHARMA\n\nCHAUKHAMBHA ORIENTALIA\nVARANASI :\n\n \n\x0c', False, True), (4, 'Publishers :\nCHAUKHAMBHA ORIENTALIA\nP.O. Chaukhambha, Post Box. No. 1032.\nGokul Bhawan, K. 37/109, Gopal Mandir Lane.\nVARANASI-221001 (India)\nTelephone : 333476 Telegram : Gokulotsav\n\nee\n\n© Chaukhambha Orientalia\nioe Fourth Edition 1998\nPrice Rs. 600-00\n\nISBN - 81-7637-011-8 ( set )\nISBN - 81-7637-014-2 ( vol-II)\n\nPrinters — Charu Printers, Varanasi-1\n\n \n\x0c', False, True)]
[(1, 're ae\n4\n\nSAMHITA\n\nENGLISH TRANSLATION)\n\nP. V. SHARMA\n\nCHAUKHAMBHA ORIENTALIA\nVARANASI :\n\n \n\x0c', False, True), (4, 'Publishers :\nCHAUKHAMBHA ORIENTALIA\nP.O. Chaukhambha, Post Box. No. 1032.\nGokul Bhawan, K. 37/109, Gopal Mandir Lane.\nVARANASI-221001 (India)\nTelephone : 333476 Telegram : Gokulotsav\n\nee\n\n© C

KeyboardInterrupt: 

In [3]:
def read_file_with_fallback_encoding(file_path):
    """Try utf-8 first, fall back to latin-1. Returns None if unreadable."""
    for encoding in ('utf-8', 'latin-1'):
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Could not read {file_path}: {e}")
            return None
    print(f"Could not decode {file_path} with any known encoding")
    return None
 

In [5]:
read_file_with_fallback_encoding('/kaggle/input/datasets/rcratos/ayurveda-texts-english/Ayurveda Dataset/ayurveda_books/Charaka_Samhita_Text_with_English_Tanslation_-_P.V._Sharma.pdf')

KeyboardInterrupt: 

In [56]:
def extract_pdf_pages(file_path):
    """
    Extract text per page from a real PDF using pdfplumber.
    Returns list of (page_num, page_text, is_estimated=False) - page numbers
    here are the ACTUAL PDF page numbers, not estimates.
    """
    pages = []
    try:
        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text() or ''
                if page_text.strip():
                    pages.append((i, page_text, False))
    except Exception as e:
        print(f"Could not extract PDF {file_path}: {e}")
        return []
    return pages

In [30]:
def split_into_pages(content, chars_per_page_estimate=1800):
    """
    Try to detect real page boundaries. Returns list of (page_num, page_text, is_estimated).
 
    Detection order:
      1. Form-feed character (\\x0c) - common when text was extracted from PDFs.
      2. "Page N" style markers on their own line.
      3. Fallback: estimate pages by character count (NOT reliable for citation,
         flagged as estimated so you know to treat it cautiously).
    """
    if '\x0c' in content:
        pages = content.split('\x0c')
        return [(i + 1, p, False) for i, p in enumerate(pages) if p.strip()]
 
    marker_pattern = re.compile(r'\n\s*Page\s+(\d+)\s*\n', re.IGNORECASE)
    matches = list(marker_pattern.finditer(content))
    if matches:
        pages = []
        for idx, m in enumerate(matches):
            page_num = int(m.group(1))
            start = m.end()
            end = matches[idx + 1].start() if idx + 1 < len(matches) else len(content)
            page_text = content[start:end]
            if page_text.strip():
                pages.append((page_num, page_text, False))
        if pages:
            return pages
 
    # Fallback: estimated pages, no real page markers found
    pages = []
    page_num = 1
    for start in range(0, len(content), chars_per_page_estimate):
        page_text = content[start:start + chars_per_page_estimate]
        if page_text.strip():
            pages.append((page_num, page_text, True))
        page_num += 1
    return pages

In [44]:
def chunk_page_paragraphs(paragraphs, page_num, page_is_estimated,
                           chunk_idx_start, global_para_start,
                           min_words=30, max_words=500):
    """
    Groups paragraphs within a page into chunks of >= min_words,
    keeping track of which paragraph number(s) each chunk spans.
    Splits any single paragraph that's larger than max_words into
    word-based sub-chunks (still tagged with the same paragraph number).
    """
    texts = []
    chunk_idx = chunk_idx_start
    global_para = global_para_start
 
    buffer = []
    buffer_para_range = None  # (first_local_para, last_local_para, first_global, last_global)
    buffer_word_count = 0
    def flush_buffer():
        nonlocal buffer, buffer_para_range, buffer_word_count, chunk_idx
        if not buffer:
            return
        combined = '\n\n'.join(buffer).strip()
        if combined and len(combined) > 10:
            texts.append({
                'chunk_idx': chunk_idx,
                'content': combined,
                'page_number': page_num,
                'page_number_estimated': page_is_estimated,
                'paragraph_number_start': buffer_para_range[0],
                'paragraph_number_end': buffer_para_range[1],
                'global_paragraph_number_start': buffer_para_range[2],
                'global_paragraph_number_end': buffer_para_range[3],
            })
            chunk_idx += 1
        buffer = []
        buffer_para_range = None
        buffer_word_count = 0
    
    for local_para_idx, para in enumerate(paragraphs, start=1):
        global_para += 1
        words = para.split()
        if not words:
            continue
    
        # Oversized single paragraph: flush what we have, then split this one on its own
        if len(words) > max_words:
            flush_buffer()
            for i in range(0, len(words), max_words):
                sub_words = words[i:i + max_words]
                sub_text = ' '.join(sub_words).strip()
                if len(sub_text) > 10:
                    texts.append({
                        'chunk_idx': chunk_idx,
                        'content': sub_text,
                        'page_number': page_num,
                        'page_number_estimated': page_is_estimated,
                        'paragraph_number_start': local_para_idx,
                        'paragraph_number_end': local_para_idx,
                        'global_paragraph_number_start': global_para,
                        'global_paragraph_number_end': global_para,
                    })
                    chunk_idx += 1
            continue
    
        if buffer_para_range is None:
            buffer_para_range = [local_para_idx, local_para_idx, global_para, global_para]
        else:
            buffer_para_range[1] = local_para_idx
            buffer_para_range[3] = global_para
    
        buffer.append(para)
        buffer_word_count += len(words)
    
        if buffer_word_count >= min_words:
            flush_buffer()
    
    flush_buffer()  # emit any remainder, even if under min_words
    
    return texts, chunk_idx, global_para


In [71]:
def process_ayurveda_texts(
    data_input_dir='/kaggle/input/datasets/rcratos/ayurveda-texts-english/Ayurveda Dataset/ayurveda_books',
    min_words=30,
    max_words=500
):
    print(f"Processing Ayurveda texts from {data_input_dir}...")
    texts = []
    all_files = []
    for root, dirs, files in os.walk(data_input_dir):
        for file_name in files:
            all_files.append((root, file_name))
    total_files = len(all_files)
    print(f"Found {total_files} files to process.")

    if TQDM_AVAILABLE:
        iterator = tqdm(all_files, desc="Processing files", unit="file")
 
    for root, dirs, files in os.walk(data_input_dir):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            relative_path = os.path.relpath(root, data_input_dir)
            category = os.path.basename(relative_path) if relative_path != '.' else 'root'
            text_name = os.path.splitext(file_name)[0]
            ext = os.path.splitext(file_name)[1].lower()

            if TQDM_AVAILABLE:
                iterator.set_postfix_str(file_name[:40])
 
            if ext == '.pdf':
                pages = extract_pdf_pages(file_path)
                if not pages:
                    continue
            else:
                content = read_file_with_fallback_encoding(file_path)
                if content is None:
                    continue
                pages = split_into_pages(content)
 
            content = read_file_with_fallback_encoding(file_path)
            if content is None:
                continue
 
            pages = split_into_pages(content)
 
            chunk_idx = 0
            global_para = 0
            for page_num, page_text, page_is_estimated,page_ocr_used  in pages:
         
                paragraphs = [p for p in page_text.split('\n\n') if p.strip()]
                page_chunks, chunk_idx, global_para = chunk_page_paragraphs(
                    paragraphs, page_num, page_is_estimated,
                    chunk_idx, global_para, min_words, max_words
                )
 
                for pc in page_chunks:
                    texts.append({
                        "id": f"{text_name}-chunk-{pc['chunk_idx']}",
                        "content": pc['content'],
                        "metadata": {
                            "source": text_name,
                            "category": category,
                            "file_path": file_name,
                            "page_number": pc['page_number'],
                            "page_number_estimated": pc['page_number_estimated'],
                            "paragraph_number_start": pc['paragraph_number_start'],
                            "paragraph_number_end": pc['paragraph_number_end'],
                            "global_paragraph_number_start": pc['global_paragraph_number_start'],
                            "global_paragraph_number_end": pc['global_paragraph_number_end'],
                        }
                    })
 
    print(f"Produced {len(texts)} chunks.")
    return texts
 

In [74]:
result = process_ayurveda_texts()

Processing Ayurveda texts from /kaggle/input/datasets/rcratos/ayurveda-texts-english/Ayurveda Dataset/ayurveda_books...
Found 21 files to process.




Processing files:   0%|          | 0/21 [00:00<?, ?file/s]

Processing files:   0%|          | 0/21 [00:00<?, ?file/s, heart_diseases.pdf]

Processing files:   0%|          | 0/21 [00:00<?, ?file/s, The_Paka_Darpanam_The_text_on_Indian_coo]

Processing files:   0%|          | 0/21 [00:00<?, ?file/s, drug_and_conmetic_act_1940.pdf]          

KeyboardInterrupt: 

In [75]:
result

[]